<a href="https://colab.research.google.com/github/azizakuanysh/DSAcademy_ML_Engineer_course/blob/main/HW04_numpy.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ДЗ 4. Операции, axis и broadcasting на данных «Дала Маркет»
Срок: **до понедельника 16.11, 19:00**.
Сдача: папка `hw04` репозитория `dsa-homework` (`hw04.ipynb`): `git add hw04` → `git commit` → `git push`, ссылку прикрепите в LMS.

Каждый ответ: **число + одна фраза для менеджера**. Циклы `for` по заказам не используем. Небольшой цикл по городам или категориям при сборке таблицы допустим.

- **[База]** обязательно для всех · **[Стандарт]** сколько получится · **[Челлендж]** для тех, кому легко.

## [Для всех] Чтение кода: без запуска
Сначала напишите ответ и «почему», потом запустите.
```python
import numpy as np
# 1
a = np.array([2, 4, 6]); print(a / 2, a // 4)
# 2
m = np.ones((3, 4)); print(m.sum(axis=1))
# 3
print(np.arange(3) + np.arange(3).reshape(3, 1))
# 4
x = np.array([1, 4, 9]); print(np.sqrt(x).sum())
# 5
print((np.arange(1, 5) ** 2).mean())
```

In [24]:
a = np.arange(3)
a
a.reshape(3,1)

array([[0],
       [1],
       [2]])

Ваши ответы и объяснения:

1. a/2 выведет все элементы строки разделенные на 2, в формате массива, а//4 это остаток деления на 4, т.е. [0,1,1]
2. np.ones - массив размерностью 3х4 состоящий из 1, axis=1 это ось х. Вывод: 4,4,4 то есть складывание вертикально. Если бы было ось 0 то ответ был бы 3,3,3,3
3. np.arange(3) - вектор-строка 0,1,2. .reshape(3,1) создает двумерный массив размерностью 3 строк и 1 столбец (вектор-столбец).
4. Сумма квадратных корней элементов х, 6
5. [1,2,3,4]**2=[1,4,8,16].mean()= 30/4=7.5

In [ ]:
# Загрузка orders.csv: в Colab нажмите «Выбрать файлы» и укажите orders.csv из LMS.
# Если файл уже лежит рядом с ноутбуком, загрузка пропускается.
import os
if not os.path.exists("orders.csv"):
    try:
        from google.colab import files
        files.upload()
    except ImportError:
        print("Положите orders.csv в одну папку с ноутбуком")
print("orders.csv на месте:", os.path.exists("orders.csv"))

orders.csv на месте: True


In [ ]:
import csv
import numpy as np

CITY = {"алматы": "Алматы", "almaty": "Алматы", "астана": "Астана", "astana": "Астана",
        "шымкент": "Шымкент", "shymkent": "Шымкент", "караганда": "Караганда",
        "актобе": "Актобе", "атырау": "Атырау"}

with open("orders.csv", encoding="utf-8") as f:
    rows = list(csv.DictReader(f))

amount = np.array([float(r["amount"].replace(" ", "")) for r in rows])
paid   = np.array([r["status"] == "оплачен" for r in rows])
days   = np.array([float(r["delivery_days"]) if r["delivery_days"] else np.nan for r in rows])
city   = np.array([CITY.get(r["city"].strip().lower(), r["city"].strip()) for r in rows])
cust   = np.array([r["customer_id"] for r in rows])
month  = np.array([r["order_date"][:7] for r in rows])      # "2026-09" или "2026-10"
cat    = np.array([r["category"] for r in rows])
oid    = np.array([r["order_id"] for r in rows])

print(amount.shape, np.unique(month))

(5463,) ['2026-09' '2026-10']


In [ ]:
# Чистые данные: без дублей (оставляем первое вхождение order_id), без TEST, только оплаченные
_, first = np.unique(oid, return_index=True)
keep = np.zeros(len(oid), dtype=bool)
keep[first] = True
clean = keep & (cust != "TEST") & paid
print("строк всего:", len(oid), "| в чистых данных:", clean.sum())

строк всего: 5463 | в чистых данных: 4739


## [База] 1. Выручка по чистым данным
Сколько строк осталось в `clean`? Выручка за сентябрь и октябрь и изменение в %. Совпадает ли с ДЗ 2?

In [ ]:
# 1

september = clean & (month == "2026-09")
october = clean & (month == "2026-10")

rev_sep = amount[september].sum()
rev_oct = amount[october].sum()

change = (rev_oct - rev_sep) / rev_sep * 100

print("Clean orders:", clean.sum())
print("September revenue:", rev_sep)
print("October revenue:", rev_oct)
print("Revenue change:", round(change, 2), "%")

Clean orders: 4739
September revenue: 35243410.0
October revenue: 34419520.0
Revenue change: -2.34 %


## [База] 2. Таблица «категория × месяц»
Соберите таблицу 5 × 2 по образцу таблицы городов с урока (категории: `np.unique(cat)`).
Какая категория приносит больше всего выручки за два месяца (`axis`)? Какая упала сильнее всех в %?

In [ ]:
# 2

categories = np.unique(cat)
months = np.array(["2026-09", "2026-10"])

category_table = np.array([
    [amount[clean & (cat == c) & (month == m)].sum()
     for m in months]
    for c in categories
])

print("Categories:", categories)
print("Columns: September, October")
print(category_table)

total_by_category = category_table.sum(axis=1)

best_category = categories[np.argmax(total_by_category)]

category_change = (
    (category_table[:, 1] - category_table[:, 0])
    / category_table[:, 0] * 100
)

worst_category = categories[np.argmin(category_change)]

print("Highest revenue category:", best_category)
print("Strongest decline:", worst_category)
print("Percentage changes:", np.round(category_change, 2))

Categories: ['Дом' 'Книги' 'Красота' 'Одежда' 'Электроника']
Columns: September, October
[[ 4997730.  5125790.]
 [ 1065300.  1140910.]
 [ 2610950.  2678990.]
 [ 9167100.  8505730.]
 [17402330. 16968100.]]
Highest revenue category: Электроника
Strongest decline: Одежда
Percentage changes: [ 2.56  7.1   2.61 -7.21 -2.5 ]


## [База] 3. Средний чек по категориям в октябре
Соберите вторую таблицу: **число** заказов по категориям и месяцам (сумма маски, а не сумма amount).
Средний чек = выручка / число заказов, одной операцией над двумя таблицами. У какой категории он самый высокий?

In [ ]:
# 3

count_table = np.array([
    [np.sum(clean & (cat == c) & (month == m))
     for m in months]
    for c in categories
])

average_check = category_table / count_table

print("Number of orders:")
print(count_table)

print("Average order value:")
print(np.round(average_check, 2))

october_avg = average_check[:, 1]

best_avg_category = categories[np.argmax(october_avg)]

print("Highest average check in October:", best_avg_category)
print("Average check:", round(october_avg.max(), 2))

Number of orders:
[[476 476]
 [265 299]
 [422 435]
 [678 652]
 [531 505]]
Average order value:
[[10499.43 10768.47]
 [ 4020.    3815.75]
 [ 6187.09  6158.6 ]
 [13520.8  13045.6 ]
 [32772.75 33600.2 ]]
Highest average check in October: Электроника
Average check: 33600.2


## [Стандарт] 4. Доля городов в выручке месяца
Используйте `table` городов с урока. Доля каждого города в выручке своего месяца: `table / table.sum(axis=0)`.
Проверьте, что каждый столбец в сумме даёт 1. Как изменилась доля Шымкента?

In [ ]:
# 4

cities = np.unique(city[clean])

table = np.array([
    [amount[clean & (city == c) & (month == m)].sum()
     for m in months]
    for c in cities
])

print("Cities:", cities)
print("City revenue table:")
print(table)

share = table / table.sum(axis=0)

print("Revenue shares:")
print(np.round(share, 4))

print("Column sums:", share.sum(axis=0))

shymkent_index = np.where(cities == "Шымкент")[0][0]

shymkent_change = (
    share[shymkent_index, 1] - share[shymkent_index, 0]
) * 100

print("Shymkent share change:", round(shymkent_change, 2), "percentage points")
table.sum(axis=0)

Cities: ['Актобе' 'Алматы' 'Астана' 'Атырау' 'Караганда' 'Шымкент']
City revenue table:
[[ 2576960.  2540420.]
 [14501070. 14643020.]
 [ 9034220.  8850530.]
 [  923990.   969860.]
 [ 3648520.  3371680.]
 [ 4558650.  4044010.]]
Revenue shares:
[[0.0731 0.0738]
 [0.4115 0.4254]
 [0.2563 0.2571]
 [0.0262 0.0282]
 [0.1035 0.098 ]
 [0.1293 0.1175]]
Column sums: [1. 1.]
Shymkent share change: -1.19 percentage points


array([35243410., 34419520.])

## [Стандарт] 5. Кто дал падение
Вклад каждого города в общее падение выручки: изменение города / общее изменение. Сколько процентов падения даёт Шымкент?
Напишите менеджеру одну фразу: где искать причину.

In [ ]:
# 5

city_change = table[:, 1] - table[:, 0]
total_change = table[:, 1].sum() - table[:, 0].sum()

contribution = city_change / total_change * 100

print("Revenue change by city:")
print(np.round(city_change, 2))

print("Contribution to total decline (%):")
print(np.round(contribution, 2))

shymkent_contribution = contribution[shymkent_index]

print("Shymkent contribution:", round(shymkent_contribution, 2), "%")

Revenue change by city:
[ -36540.  141950. -183690.   45870. -276840. -514640.]
Contribution to total decline (%):
[  4.44 -17.23  22.3   -5.57  33.6   62.46]
Shymkent contribution: 62.46 %


## [Стандарт] 6. Исправьте код от AI
AI написал код «доля сентября и октября внутри каждого города». Код падает. Объясните, почему, и исправьте.
```python
share = table / table.sum(axis=1)
```

In [ ]:
# 6

share = table / table.sum(axis=1, keepdims=True)

print("September and October shares within each city:")
print(np.round(share, 4))

print("Row sums:", share.sum(axis=1))

## [Челлендж] 7. Стандартизация суммы заказа
Для чистых оплаченных заказов посчитайте z-оценку: `z = (x - x.mean()) / x.std()`.
Сколько заказов с |z| > 3 и какая это доля? Проверьте, что у `z` среднее ≈ 0 и std = 1.

In [ ]:
# 7

## [Челлендж] 8. Масштабирование признаков (заготовка для ML)
Соберите матрицу `X` из двух столбцов: `amount` и `days` для чистых заказов без пропусков в `days` (`np.column_stack`).
Приведите каждый столбец к диапазону [0, 1] одной строкой: `(X - X.min(axis=0)) / (X.max(axis=0) - X.min(axis=0))`.
Проверьте min и max каждого столбца. Объясните своими словами, зачем модели нужны признаки в одном масштабе.

In [ ]:
# 8